# nanochat.cpp session setup (Kaggle)

Prepare one Kaggle session for `nanochat.cpp` and the `pi` agent.

This notebook loads the API keys from Kaggle secrets. It then clones the
repository on the first run, or pulls the latest revision on every later run.

Run this notebook before a training notebook or an agent notebook. Set the
accelerator to T4 and turn Internet on.

## API keys

Kaggle secrets hold the credentials. Each secret name becomes an environment
variable. `HF_TOKEN` lets the data download run without a prompt. The other
keys authenticate the `pi` model provider. The notebook reports a
missing secret. The run continues.

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

# Each Kaggle secret name becomes an environment variable of the same name.
API_KEY_NAMES = [
    "HF_TOKEN",
    "GEMINI_API_KEY",
    "OPENROUTER_API_KEY",
    "DEEPSEEK_API_KEY",
    "OPENCODE_API_KEY",
]

secrets = UserSecretsClient()
for name in API_KEY_NAMES:
    try:
        os.environ[name] = secrets.get_secret(name)
        print("set", name)
    except Exception:
        print("missing", name)

## Repository

Clone the repository on the first run. On every later run, fetch, check out
`CODE_REF`, and pull. `CODE_REF` is `None` for the default branch. The pull
keeps local edits, so a divergent working tree is not overwritten.

In [ ]:
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/ming6ao/nanochat.cpp"
REPO = Path("/kaggle/working/nanochat.cpp")

# The revision to run. `None` uses the repository default branch.
CODE_REF = None

if not (REPO / ".git").is_dir():
    subprocess.run(["git", "clone", REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO), "fetch", "--all", "--prune"],
                   check=True)
if CODE_REF:
    subprocess.run(["git", "-C", str(REPO), "checkout", CODE_REF], check=True)
# A detached head has no upstream, so the pull is best effort.
subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=False)

# Make the repository packages importable before the setup import.
sys.path[:0] = [str(REPO / "python"), str(REPO / "notebooks")]

revision = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "--short", "HEAD"],
    capture_output=True, text=True, check=True).stdout.strip()
print("repository:", REPO)
print("revision:", revision)

## Host

`kaggle_setup.setup` runs `tools/kaggle/bootstrap.sh`, reads the host
contract, and builds the shared library for the T4 (`sm_75`). The bootstrap
also installs `pi` and Node.js.

In [ ]:
import kaggle_setup

host = kaggle_setup.setup(url=REPO_URL, dest=REPO, ref=CODE_REF, arch="sm_75")
print("library:", host.library)
print("build key:", host.build_key)

pi_version = subprocess.run(["pi", "--version"], capture_output=True,
                            text=True).stdout.strip()
devices = subprocess.run(
    ["nvidia-smi", "--query-gpu=index,name,memory.total",
     "--format=csv,noheader"], capture_output=True, text=True).stdout.strip()
print("pi:", pi_version)
print("devices:")
print(devices)

## Pi check

Run one small prompt through the agent. A provider error raises an exception
and prints the cause. An empty answer can no longer pass in silence.

The cell pins the provider and the bare model ID through the `PI_PROVIDER`
and `PI_MODEL` variables. Set the `PI_AGENT_PROVIDER` and `PI_AGENT_MODEL`
environment variables to change them. Tool activity updates one
cumulative line in place.

In [ ]:
import importlib

import pi_agent

# A pull does not reload an already imported module. Reload it, so a
# new commit takes effect without a kernel restart, then bind the name.
importlib.reload(pi_agent)
from pi_agent import run_pi

# Pin the provider and the bare model ID. Without them, Pi selects a
# provider automatically and can reject the model with
# "Model access is disabled".
PI_PROVIDER = os.environ.get("PI_AGENT_PROVIDER", "opencode-go")
PI_MODEL = os.environ.get("PI_AGENT_MODEL", "deepseek-v4.1-flash")

run_pi("Reply with the single word READY.", cwd=REPO,
       provider=PI_PROVIDER, model=PI_MODEL)